In [8]:
import re
import pandas as pd
import copy
import numpy as np
import evaluate
from docx import Document
import os
import unittest
from datasets import load_dataset
import pickle
import matplotlib.pyplot as plt
from bleurt import score
class DataLoader:
    """
    Loader for benchmarking datasets to ensure universal formatting. To be used in conjunction with DyslexiaInjector.
    ...
    Attributes
    ----------
    path: str
        Path to csv, txt or docx file of the data. In the case of CSV there should only be 1 column
    data: list
        A list of striings
    dataset_name: str
        Name of the dataset that is used when saving the data
    ...
    Methods
    -------
    parse_txt(path)
        Parses a txt file and returns a list of strings
    fix_format(sentence)
        Fixes the formatting of a sentence
    save_as_txt(path)
        Saves the data as a txt file
    save_as_csv(path)
        Saves the data as a csv file
    save_as_docx(path)
        Saves the data as a docx file
    get_data()
        Returns the data
    create_deepcopy()
        Returns a deepcopy of the DataLoader instance
    get_name()
        Returns the dataset name
    get_number_of_sentences()
        Returns the number of sentences in the data
    get_number_of_words()
        Returns the number of words in the data
    get_number_of_letters()
        Returns the number of letters in the data
    edit_distance(reference_sentence, sentence)
        Returns the number of edits required to transform reference_sentence into sentence at word level
        edits include insertions, deletions and substitutions
        based on levenshtein distance
        also returns a dictionary of substitutions, insertions and deletions
    get_edit_distance(reference, manual_wer=False)
        Returns the number of edits required to transform data into reference at word level, substitutions, insertions and deletions the associated dictionaries
        and the WER (withouth alignment) if manual_wer is set to True
    get_individual_edit_distance(reference)
        Returns the number of edits required to transform data into reference at word level for each individual sentence
    combine_nested_dict(dict1, dict2)
        Combines two nested dictionaries
    combine_dicts(dict1, dict2)
        Combines two dictionaries
    get_bleue_score(reference)
        Returns bleu score of the data against a reference
    get_wer(reference)
        Returns the Word Error Rate (WER) of the data against a reference. With word alignment
    get_bert_score(reference)
        Returns the BERT Score similarity score of the data against a reference
    get_LaBSE(reference, model=None, tokenizer=None)
        Returns the LaBSE similarity score of the data against a reference which is a l2 norm between the reference and target sentences score.
        Score of 1 means the sentences are identical, closer to 0 means they are less similar semantically.
    ...

    Usage
    -------
    >>> from datasets import load_dataset
    >>> from DataLoader import DataLoader
    >>> dataset_wmt_enfr = load_dataset("wmt14",'fr-en', split='test')
    >>> to_translate = []
    >>> for i in range(len(dataset_wmt_enfr)):
    >>>     to_translate.append(dataset_wmt_enfr[i]['translation']['en'])
    >>> loader = DataLoader(data=to_translate, dataset_name="wmt14_enfr")
    >>> loader.save_as_txt("wmt14_enfr.txt")
    We can also use the text file to create a new DataLoader instance
    >>> loader2 = DataLoader(path="wmt14_enfr.txt", dataset_name="wmt14_enfr")
    """
    # Constructor
    def __init__(self, path=None, data=None, dataset_name=""):
        self.dataset_name = dataset_name
        if data is None and path is not None:
            #check path to see if file is txt or csv
            file_type = path.split(".")[-1]
            if file_type == "txt":
                self.data = self.parse_txt(path)
                self.data = [self.fix_format(sentence) for sentence in self.data]
            elif file_type == "csv":
                self.data = pd.read_csv(path, header=None)
                self.data = self.data[0].tolist()
                #fix any formatting issues
                self.data = [self.fix_format(sentence) for sentence in self.data]
            elif file_type == "docx":
                doc = Document(path)
                self.data = [self.fix_format(paragraph.text) for paragraph in doc.paragraphs]
            else:
                raise Exception("Invalid file type")
        elif data is not None:
            #check if data is a list or a df
            if isinstance(data, list):
                #format each sentence in data
                self.data = [self.fix_format(sentence) for sentence in data]
            else:
                raise Exception("Invalid data type, please pass in a list of sentences")
        else:
            raise Exception("Please pass in a path or data")

    def parse_txt(self, path):
        output = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                output.append(self.fix_format(line))
        return output
                
    def fix_format(self, sentence):
        #remove spacing before punctuation
        sentence = re.sub(r'\s([?.!,"](?:\s|$))', r'\1', sentence)
        #replace any double spaces with single space
        sentence = re.sub(r'\s+', ' ', sentence)
        #remove any leading or trailing spaces
        sentence = sentence.strip()
        #make all quotes (german and french) english double quotes
        sentence = re.sub(r'«|»|„|“', '"', sentence)
        #make all single quotes english single quotes
        sentence = re.sub(r'‘|’', "'", sentence)
        #make all french guillemets english double quotes
        sentence = re.sub(r'‹|›', '"', sentence)
        #if sentence begins and ends with quotes and there are only two, remove them
        if sentence[0] == '"' and sentence[-1] == '"' and sentence.count('"') == 2:
            sentence = sentence[1:-1]
        elif sentence[0] == "'" and sentence[-1] == "'" and sentence.count("'") == 2:
            sentence = sentence[1:-1]
        return sentence

    def save_as_txt(self, path):
        with open(path, "w", encoding="utf-8") as f:
            for sentence in self.data:
                f.write(f"{sentence}\n")
        print(f"Saved {self.dataset_name} to {path}")
        return
    
    def save_as_csv(self, path):
        df = pd.DataFrame(self.data)
        df.to_csv(path, index=False, header=False, encoding='utf-8')
        print(f"Saved {self.dataset_name} to {path}")
        return
    
    def save_as_docx(self, path):
        document = Document()
        for sentence in self.data:
            document.add_paragraph(sentence)
        document.save(path)
        print(f"Saved {self.dataset_name} to {path}")
        return

    def get_data(self):
        return self.data

    def create_deepcopy(self):
        return DataLoader(data=copy.deepcopy(self.data), dataset_name=self.dataset_name)
        
    def get_name(self):
        return self.dataset_name

    def get_number_of_sentences(self):
        return len(self.data)
    
    def get_number_of_words(self):
        return sum([len(sentence.split()) for sentence in self.data])
    
    def get_number_of_letters(self):
        #need to ensure we only count letters and not punctuation
        return sum([len(re.sub(r'[^\w\s]','',sentence)) for sentence in self.data])

    def edit_distance(reference_sentence, sentence):
        """
        Returns the number of edits required to transform reference_sentence into sentence at word level
        edits include insertions, deletions and substitutions
        based on levenshtein distance
        also returns a dictionary of substitutions, insertions and deletions
        """
        substitutions = 0
        insertions = 0
        deletions = 0
        substitution_dict = {}
        insertion_dict = {}
        deletion_dict = {}
        #remove punctuation and split into words
        sentence = re.sub(r'[^\w\s]','',sentence).lower().split()
        reference_sentence = re.sub(r'[^\w\s]','',reference_sentence).lower().split()
        #create matrix
        matrix = np.zeros((len(reference_sentence)+1,len(sentence)+1))
        #fill in first row and column
        for i in range(len(reference_sentence)+1):
            matrix[i][0] = i
        for j in range(len(sentence)+1):
            matrix[0][j] = j
        #fill in rest of matrix
        for i in range(1,len(reference_sentence)+1):
            for j in range(1,len(sentence)+1):
                if sentence[j-1] == reference_sentence[i-1]:
                    matrix[i][j] = matrix[i-1][j-1]
                else:
                    matrix[i][j] = min(matrix[i-1][j-1], matrix[i-1][j], matrix[i][j-1])+1
        #backtrack to find edits
        i = len(reference_sentence)
        j = len(sentence)
        while i > 0 and j > 0:
            if sentence[j-1] == reference_sentence[i-1]:
                i -= 1
                j -= 1
            else:
                if matrix[i][j] == matrix[i-1][j-1]+1:
                    substitutions += 1
                    if reference_sentence[i-1] not in substitution_dict:
                        substitution_dict[reference_sentence[i-1]] = {sentence[j-1]:1}
                    else:
                        if sentence[j-1] not in substitution_dict[reference_sentence[i-1]]:
                            substitution_dict[reference_sentence[i-1]][sentence[j-1]] = 1
                        else:
                            substitution_dict[reference_sentence[i-1]][sentence[j-1]] += 1
                    i -= 1
                    j -= 1
                elif matrix[i][j] == matrix[i-1][j]+1:
                    deletions += 1
                    if reference_sentence[i-1] not in deletion_dict:
                        deletion_dict[reference_sentence[i-1]] = 1
                    else:
                        deletion_dict[reference_sentence[i-1]] += 1
                    i -= 1
                elif matrix[i][j] == matrix[i][j-1]+1:
                    insertions += 1
                    if sentence[j-1] not in insertion_dict:
                        insertion_dict[sentence[j-1]] = 1
                    else:
                        insertion_dict[sentence[j-1]] += 1
                    j -= 1
        while i > 0:
            deletions += 1
            if reference_sentence[i-1] not in deletion_dict:
                deletion_dict[reference_sentence[i-1]] = 1
            else:
                deletion_dict[reference_sentence[i-1]] += 1
            i -= 1
        while j > 0:
            insertions += 1
            if sentence[j-1] not in insertion_dict:
                insertion_dict[sentence[j-1]] = 1
            else:
                insertion_dict[sentence[j-1]] += 1
            j -= 1
        distance = substitutions+insertions+deletions
        return substitutions, insertions, deletions, substitution_dict, insertion_dict, deletion_dict, distance
        
    def get_edit_distance(self, reference, manual_wer=False):
        """
        Returns the number of edits required to transform data into reference at word level, substitutions, insertions and deletions the associated dictionaries
        and the WER (withouth alignment) if manual_wer is set to True
        """
        if type(reference) == list:
            substitutions = 0
            insertions = 0
            deletions = 0
            all_sub = {}
            all_ins = {}
            all_del = {}
            distance = 0
            for i in range(len(self.data)):
                sub, ins, dele, substitution_dict, insertion_dict, deletion_dict, dist = DataLoader.edit_distance(reference[i], self.data[i], )
                all_sub = self.combine_nested_dict(all_sub, substitution_dict)
                all_ins = self.combine_dicts(all_ins, insertion_dict)
                all_del = self.combine_dicts(all_del, deletion_dict)
                substitutions += sub
                insertions += ins
                deletions += dele
                distance += dist
            if manual_wer:
                return substitutions, insertions, deletions, all_sub, all_ins, all_del, distance, distance/(sum([len(sentence.split()) for sentence in reference]))
            return substitutions, insertions, deletions, all_sub, all_ins, all_del, distance
        elif type(reference) == DataLoader:
            return self.get_edit_distance(reference.get_data(), manual_wer=manual_wer)
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")

    def get_individual_edit_distance(self, reference):
        """
        Returns the number of edits required to transform data into reference at word level for each individual sentence
        """
        if type(reference) == list:
            output = []
            for i in range(len(self.data)):
                sub, ins, dele, substitution_dict, insertion_dict, deletion_dict, distance = DataLoader.edit_distance(reference[i], self.data[i], )
                output.append((sub, ins, dele, substitution_dict, insertion_dict, deletion_dict, distance))
            return output
        elif type(reference) == DataLoader:
            return self.get_individual_edit_distance(reference.get_data())
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")       

    def combine_nested_dict(self, dict1, dict2):
        for key in dict2:
            if key not in dict1:
                dict1[key] = dict2[key]
            else:
                for key2 in dict2[key]:
                    if key2 not in dict1[key]:
                        dict1[key][key2] = dict2[key][key2]
                    else:
                        dict1[key][key2] += dict2[key][key2]
        return dict1
    
    def combine_dicts(self, dict1, dict2):
        for key in dict2:
            if key not in dict1:
                dict1[key] = dict2[key]
            else:
                dict1[key] += dict2[key]
        return dict1

    def get_bleue_score(self, reference):
        #returns bleu score of the data against a reference
        bleu = evaluate.load("bleu")
        if type(reference) == list:
            return bleu.compute(predictions=self.data, references=reference)
        elif type(reference) == DataLoader:
            return bleu.compute(predictions=self.data, references=reference.get_data())
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")

    def get_wer(self, reference):
        """
        Returns the Word Error Rate (WER) of the data against a reference. With word alignment
        """
        wer = evaluate.load("wer")
        if type(reference) == list:
            return wer.compute(predictions=self.data, references=reference)
        elif type(reference) == DataLoader:
            return wer.compute(predictions=self.data, references=reference.get_data())
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")


    def get_bert_score(self, reference, lang="fr"):
        """
        Returns the BERT Score similarity score of the data against a reference.
        """
        bert = evaluate.load("bertscore")
        if type(reference) == list:
            return bert.compute(predictions=self.data, references=reference, lang=lang)
        elif type(reference) == DataLoader:
            return bert.compute(predictions=self.data, references=reference.get_data(), lang=lang)
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")

    def get_LaBSE(self, reference, model=None, tokenizer=None):
        """
        Returns the LaBSE similarity score of the data against a reference which is a l2 norm between the reference and target sentences score.
        Score of 1 means the sentences are identical, closer to 0 means they are less similar semantically.
        """
        if model is None:
            model = BertModel.from_pretrained("setu4993/LaBSE")
        if tokenizer is None:
            tokenizer = BertTokenizerFast.from_pretrained("setu4993/LaBSE")
        if type(reference) == list:
            pass
        elif type(reference) == DataLoader:
            reference = reference.get_data()
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")
        target = self.data
        reference_inputs = tokenizer(reference, return_tensors="pt", padding=True).to("cuda")
        target_inputs = tokenizer(target, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            reference_outputs = model(**reference_inputs)
            target_outputs = model(**target_inputs)
        reference_embeddings = reference_outputs.pooler_output
        target_embeddings = target_outputs.pooler_output
        return self.similarity(reference_embeddings, target_embeddings)
    
    def get_bleurt(self, reference, scorer = None):
        """
        BLEURT-20 is required and can be downloaded via https://github.com/google-research/bleurt
        This is the most up to date version of BLEURT and is multilingual
        Returns the BLEURT similarity score of the data against a reference.
        """
        if scorer is None:
            try:
                scorer = score.BleurtScorer("BLEURT-20")
            except:
                raise Exception("BLEURT-20 not found")
        if type(reference) == list:
            scores = scorer.score(references = reference, candidates = self.data)
        elif type(reference) == DataLoader:
            scores = scorer.score(references = reference.get_data(), candidates = self.data)
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")
        return scores
    def get_COMET(self, reference, source):
        """
        Returns the COMET similarity score of the data against a reference and a source.
        Source is the original sentence and reference is the translation
        """
        comet = evaluate.load("comet")
        if type(reference) == list:
            return comet.compute(predictions=self.data, references=reference, sources=source)
        elif type(reference) == DataLoader:
            return comet.compute(predictions=self.data, references=reference.get_data(), sources=source.get_data())
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")

In [9]:
# v2_raw_dyslexia_corpus_en = DataLoader(path="r_Dyslexia_Text\default_files\\raw\\v2_raw_reddit_text.txt", dataset_name="v2_reddit_dyslexia")
# v2_corrected_dyslexia_corpus_en = DataLoader(path="r_Dyslexia_Text\default_files\\corrected\\v2_corrected_reddit_text.txt", dataset_name="v2_corrected_reddit_dyslexia")
# v1_raw_dyslexia_corpus_en = DataLoader(path="r_Dyslexia_Text\default_files\\raw\\raw_reddit_text.txt", dataset_name="v1_raw_reddit_dyslexia")
# v1_corrected_dyslexia_corpus_en = DataLoader(path="r_Dyslexia_Text\default_files\\corrected\\corrected_reddit_text.txt", dataset_name="v1_corrected_reddit_dyslexia")
combined_raw_dyslexia_corpus_en = DataLoader(path="r_Dyslexia_Text\default_files\\raw\\combined_raw_reddit_text.txt", dataset_name="combined_raw_reddit_dyslexia")
combined_corrected_dyslexia_corpus_en = DataLoader(path="r_Dyslexia_Text\default_files\\corrected\\combined_corrected_reddit_text.txt", dataset_name="combined_corrected_reddit_dyslexia")

In [10]:
import matplotlib.pyplot as plt
import os
import sys  
import difflib  
from difflib import unified_diff  

In [11]:
from collections import Counter
def compare_sentences(s1, s2):
    '''s1 is the input sentence, s2 is the reference sentence'''
    s1 = s1.strip().lower().split()
    s2 = s2.strip().lower().split()

    #remove any punctuation
    s1 = [re.sub(r'[^\w\s]','',word) for word in s1]
    s2 = [re.sub(r'[^\w\s]','',word) for word in s2]
    words_in_this_sentence = {}
    for i in difflib.ndiff(s1, s2):
        if i[0] == ' ': continue
        if i[0] == '-':
            if words_in_this_sentence.get(i[2:]) is None:
                words_in_this_sentence[i[2:]] = 1
            else:
                words_in_this_sentence[i[2:]] += 1
        elif i[0] == '+':
            if words_in_this_sentence.get(i[2:]) is None:
                words_in_this_sentence[i[2:]] = -1
            else:
                words_in_this_sentence[i[2:]] -= 1
        else:
            #ignore any other symbols
            continue

    #remove all values in the dict that are 0
    words_in_this_sentence = {k: v for k, v in words_in_this_sentence.items() if v != 0}
    return words_in_this_sentence

def compare_df_setence_level(df, reference):
    '''df is a dataframe with the translated sentences, reference is a DataLoader instance with the reference sentences'''
    df = df.get_data()
    reference = reference.get_data()
    totals = Counter()
    for i in range(len(df)):
       totals.update(Counter(compare_sentences(df[i], reference[i])))
    return totals

In [12]:
#differ compare
differ = difflib.Differ()

s1 = "\n".join(combined_raw_dyslexia_corpus_en.get_data())
s2 = "\n".join(combined_corrected_dyslexia_corpus_en.get_data())
for line in differ.compare(s2.splitlines(keepends=True), s1.splitlines(keepends=True)):
    print(line, end="")
#compare sentences

- Well I got diagnosed with dyslexia a year ago when I was 18... And I do have some struggles with math too, like I never got the 1x1's in my brain, take longer than most, can't imagine numbers in my brain and calculate them at the same time so have to see the numbers on paper and do the math on paper. But I don't know if it's just being kind of bad in math or actually having it. My parents I still live with don't even believe in Dyslexia... say I just got it because my teacher in 1st grade taught me wrong. and from childhood I was actually good at math and just struggled at some parts. A test is kind of expensive... and hard to find where I can get tested. I don't know what I should do... (Probably a rant, sorry)
?      ^                                             ^           ^^^^^                -                           ^                                                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                               

In [13]:
from IPython import display

In [14]:
# a = open(r"r_Dyslexia_Text\\default_files\\raw\\raw_reddit_text.txt", "r").readlines()
# b = open(r"r_Dyslexia_Text\\default_files\\corrected\\corrected_reddit_text.txt", "r").readlines()
# diff = difflib.HtmlDiff(tabsize=2, wrapcolumn=90)
# with open("test.html", "w") as f:
#     f.write(diff.make_file(fromlines=a,tolines=b, fromdesc="raw text", todesc="corrected"))

In [15]:
# v2_raw_dyslexia_corpus_en = DataLoader(path="r_Dyslexia_Text\default_files\\raw\\v2_raw_reddit_text.txt", dataset_name="v2_reddit_dyslexia")
# v2_corrected_dyslexia_corpus_en = DataLoader(path="r_Dyslexia_Text\default_files\\corrected\\v2_corrected_reddit_text.txt", dataset_name="v2_corrected_reddit_dyslexia")
# v1_raw_dyslexia_corpus_en = DataLoader(path="r_Dyslexia_Text\default_files\\raw\\raw_reddit_text.txt", dataset_name="v1_raw_reddit_dyslexia")
# v1_corrected_dyslexia_corpus_en = DataLoader(path="r_Dyslexia_Text\default_files\\corrected\\corrected_reddit_text.txt", dataset_name="v1_corrected_reddit_dyslexia")

In [16]:
print(f"Number of letters in raw v1: {combined_raw_dyslexia_corpus_en.get_number_of_letters()}")
print(f"Number of of words in raw v1: {combined_raw_dyslexia_corpus_en.get_number_of_words()}")
print(f"Number of posts in raw v1: {combined_raw_dyslexia_corpus_en.get_number_of_sentences()}")

print(f"Number of letters in corrected v1: {combined_corrected_dyslexia_corpus_en.get_number_of_letters()}")
print(f"Number of of words in corrected v1: {combined_corrected_dyslexia_corpus_en.get_number_of_words()}")
print(f"Number of posts in corrected v1: {combined_corrected_dyslexia_corpus_en.get_number_of_sentences()}")



Number of letters in raw v1: 19902
Number of of words in raw v1: 3919
Number of posts in raw v1: 33
Number of letters in corrected v1: 19938
Number of of words in corrected v1: 3934
Number of posts in corrected v1: 33


In [17]:
#172 sentences in v1 corrected reddit text
#64 sentences in v2 corrected reddit text
#total of 236 sentences

In [18]:
combined_WER = combined_raw_dyslexia_corpus_en.get_wer(combined_corrected_dyslexia_corpus_en)
combined_WER

0.12582613116420946

In [19]:
# import numpy as np

# def levenshtein_distance(s1, s2):
#     len_s1 = len(s1) + 1
#     len_s2 = len(s2) + 1

#     # Create a matrix to hold the distances
#     matrix = np.zeros((len_s1, len_s2))

#     # Initialize the matrix
#     for i in range(len_s1):
#         matrix[i][0] = i
#     for j in range(len_s2):
#         matrix[0][j] = j

#     # Calculate the Levenshtein distance
#     for i in range(1, len_s1):
#         for j in range(1, len_s2):
#             if s1[i-1] == s2[j-1]:
#                 cost = 0
#             else:
#                 cost = 1
#             matrix[i][j] = min(matrix[i-1][j] + 1,     # deletion
#                                matrix[i][j-1] + 1,     # insertion
#                                matrix[i-1][j-1] + cost)  # substitution

#     return matrix[len_s1 - 1][len_s2 - 1], matrix

# def words_modified_percentage(s1, s2):
#     words1 = s1.split()
#     words2 = s2.split()
#     distance, matrix = levenshtein_distance(words1, words2)
#     insertions, deletions, substitutions = count_operations(matrix, words1, words2)
    
#     total_words = len(words1)
#     modified_words = insertions + deletions + substitutions
    
#     percentage_modified = (modified_words / total_words) * 100
    
#     return percentage_modified, modified_words, total_words

# def count_operations(matrix, s1, s2):
#     i, j = len(s1), len(s2)
#     insertions, deletions, substitutions = 0, 0, 0

#     while i > 0 or j > 0:
#         current = matrix[i][j]
#         if i > 0 and j > 0 and (matrix[i-1][j-1] < current):
#             if s1[i-1] != s2[j-1]:
#                 substitutions += 1
#             i -= 1
#             j -= 1
#         elif i > 0 and (matrix[i-1][j] < current):
#             deletions += 1
#             i -= 1
#         elif j > 0 and (matrix[i][j-1] < current):
#             insertions += 1
#             j -= 1
#         else:
#             i -= 1
#             j -= 1

#     return insertions, deletions, substitutions


In [20]:
from jiwer import compute_measures
# Example arrays with sentences
array1 = combined_raw_dyslexia_corpus_en.get_data()
array2 = combined_corrected_dyslexia_corpus_en.get_data()

total_insertions, total_deletions, total_substitutions = 0, 0, 0
modified_words, total_words = 0, 0

for s1, s2 in zip(array1, array2):
    measures = compute_measures(s1, s2)
    distance = measures['wer'] * len(s2.split())  # Calculate Levenshtein distance approximation
    insertions = measures['insertions']
    deletions = measures['deletions']
    substitutions = measures['substitutions']

    total_insertions += insertions
    total_deletions += deletions
    total_substitutions += substitutions


print(f'Total insertions: {total_insertions}, Total deletions: {total_deletions}, Total substitutions: {total_substitutions}')


wer = (total_insertions + total_deletions + total_substitutions) / 3934
print("Manual WER: ", wer)


Total insertions: 56, Total deletions: 41, Total substitutions: 398
Manual WER:  0.12582613116420946


In [21]:
#reading in our dictionaries
import pickle

file = open("./dict/pedler_dict.pickle", "rb")
pedler_dict = pickle.load(file)
pedler_dict

{'ably': ['able'],
 'able': ['ably'],
 'aboard': ['abroad'],
 'abroad': ['aboard'],
 'abut': ['about'],
 'about': ['abut'],
 'accept': ['except'],
 'except': ['accept', 'excerpt'],
 'acceptably': ['acceptable'],
 'acceptable': ['acceptably'],
 'accursed': ['accused'],
 'accused': ['accursed'],
 'ace': ['ache', 'acre', 'axe', 'age'],
 'ache': ['ace', 'acre', 'axe'],
 'acre': ['ace', 'ache', 'axe'],
 'axe': ['ace', 'ache', 'acre', 'age'],
 'aces': ['acres', 'axes', 'ages', 'axis'],
 'acres': ['aces', 'axes'],
 'axes': ['aces', 'acres', 'ages', 'axis'],
 'ached': ['arched'],
 'arched': ['ached'],
 'aches': ['arches'],
 'arches': ['aches'],
 'aching': ['arching'],
 'arching': ['aching'],
 'action': ['auction'],
 'auction': ['action'],
 'actions': ['auctions'],
 'auctions': ['actions'],
 'ad': ['add', 'ado', 'aid'],
 'add': ['ad', 'ado', 'aid'],
 'ado': ['ad', 'add', 'aid'],
 'aid': ['ad', 'add', 'ado', 'aide'],
 'adapt': ['adept'],
 'adept': ['adapt'],
 'addressees': ['addresses'],
 'addre

In [22]:
file = open("./dict/homophones_dict.pickle", "rb")
homophones_dict = pickle.load(file)
homophones_dict

{'a': ['uh'],
 'the': ['thee'],
 'capital': ['capitol'],
 'flew': ['flu', 'flue'],
 'past': ['passed'],
 'there': ['their', "they're"],
 'to': ['too', 'two'],
 'box': ['bocks'],
 'in': ['inn'],
 'your': ['yore', "you're"],
 'as': ['ass', 'asse'],
 'road': ['rode', 'rowed'],
 'find': ['fined'],
 'cash': ['cache'],
 'are': ['air', 'aire', 'ayre', 'ere', 'err', 'eyre', 'heir'],
 'see': ['c', 'cee', 'sea'],
 'by': ['bye', 'bi', 'buy'],
 'which': ['wich', 'witch'],
 'for': ['fore', 'four'],
 'intense': ['intents'],
 'have': ['halve'],
 'use': ['ewes', 'yews'],
 'you': ['ewe', 'yew'],
 'where': ['ware', 'wear', 'weir'],
 'tax': ['tacks'],
 'tea': ['t', 'tee', 'ti'],
 'too': ['to', 'two'],
 'raising': ['rasing', 'razing'],
 'while': ['wile'],
 "can't": ['cant'],
 'whether': ['weather', 'wether'],
 'not': ['knot'],
 'waiting': ['weighting'],
 'per': ['purr'],
 'roll': ['role'],
 'some': ['sum'],
 'must': ['mussed'],
 'our': ['hour'],
 'we': ['wee', 'whee'],
 'might': ['mite'],
 'choose': ['che

In [23]:
file = open("./dict/confusing_letters_dict.pickle", "rb")
confusing_letters_dict = pickle.load(file)
confusing_letters_dict

{'a': ['e', 'i', 'o', 'u'],
 'b': ['d', 'p', 'q'],
 'c': ['e', 's', 'k'],
 'd': ['b', 'p', 'q'],
 'e': ['a', 'o', 'u', 'i'],
 'f': ['t'],
 'h': ['n'],
 'i': ['j', 'o', 'u', 'a', 'e'],
 'j': ['i'],
 'k': ['x'],
 'l': ['i'],
 'm': ['n', 'w'],
 'n': ['h', 'm', 'u'],
 'o': ['a', 'e'],
 'p': ['b', 'd', 'q'],
 'q': ['b', 'd', 'p'],
 'r': ['n'],
 's': ['c'],
 't': ['f'],
 'u': ['i', 'v', 'n'],
 'v': ['u', 'w'],
 'w': ['v', 'm'],
 'x': ['k'],
 'y': ['v', 'i'],
 'z': ['s']}

In [ ]:
# combined_raw_dyslexia_corpus_en
# combined_corrected_dyslexia_corpus_en


#calculate percentage of words modified
words_modified = 0
total_words = combined_corrected_dyslexia_corpus_en.get_number_of_words()
arr1 = combined_raw_dyslexia_corpus_en.get_data()
arr2 = combined_corrected_dyslexia_corpus_en.get_data()
for s1, s2 in zip(arr1, arr2):
    s1 = s1.split()
    s2 = s2.split()
    for i in range(len(s2)):
        if i >= len(s1):
            break
        if s1[i].lower() != s2[i].lower():
            print(s1[i], s2[i])
            words_modified += 1

print(f"There were {words_modified} words modified")
print(f"There are a total of {total_words} words")
print(f"Percentage of words modified: {words_modified / total_words * 100}%")
#Not as trivial to calculate

stuggles struggles
Imagen imagine
caculate calculate
paper..But paper.
i But
dont I
know don't
if know
its if
just it's
being just
kind being
of kind
bad of
in bad
math in
or math
actually or
having actually
it. having
My it.
parents My
i parents
still I
live still
with live
dont with
even don't
believe even
in believe
Dylexia... in
say Dyslexia...
i say
just I
got just
it got
because it
my because
teacher my
in teacher
1st in
grade 1st
teached grade
me taught
wrong.. me
and wrong.
from and
childhood from
i childhood
was I
actually was
good actually
in good
math at
just math
struggled and
at just
some struggled
parts.A at
test some
is parts.
kind A
of test
expensive.. is
and kind
hard of
to expensive...
find and
where hard
i to
can find
get where
tested I
i can
dont get
know tested.
what I
i don't
should know
do.. what
(Probally I
a should
rant, do...
sorry) (Probably
comparisson comparison
exlpain explain
it:There it:
is there
a is
person a
that person
misses that
part misses
of part


In [34]:
#Calcualted how many pedler confusions there are 
import re


pedler_confusions = 0
for s1, s2 in zip(arr1, arr2):
    #lower case and remove punctuation
    s1 = re.sub(r'[^\w\s]','',s1).lower().split()
    s2 = re.sub(r'[^\w\s]','',s2).lower().split()
    #give a 3 word buffer zone
    for i in range(len(s1)):
        if i >= len(s2):
            break
        if s1[i] == s2[i]:
            continue
        if s1[i] in pedler_dict:
            if s2[i] in pedler_dict[s1[i]]:
                pedler_confusions += 1
                print(f"Confused word: {s1[i]} and {s2[i]}")
                continue
            elif i > 0 and i < len(s2) - 1 and s1[i] in pedler_dict:
                if s2[i+1] in pedler_dict[s1[i]]:
                    pedler_confusions += 1
                    print(f"Confused word: {s1[i]} and {s2[i+1]}")
                    continue
                elif s2[i-1] in pedler_dict[s1[i]]:
                    pedler_confusions += 1
                    print(f"Confused word: {s1[i]} and {s2[i-1]}")
                    continue
            
print(f"There were {pedler_confusions} pedler confusions")
print(f"The percentage of pedler confusions is {pedler_confusions / total_words * 100}%")

Confused word: barley and barely
Confused word: know and knew
Confused word: they and their
Confused word: or and are
Confused word: are and or
Confused word: your and you
Confused word: you and your
Confused word: your and you
Confused word: an and and
Confused word: an and and
Confused word: to and do
Confused word: life and live
Confused word: whit and with
Confused word: to and do
Confused word: do and to
Confused word: am and and
Confused word: an and and
Confused word: of and off
Confused word: tens and tense
Confused word: herd and heard
Confused word: slip and sleep
Confused word: seams and seems
Confused word: advice and advise
Confused word: the and there
Confused word: came and come
Confused word: my and me
Confused word: to and too
Confused word: an and and
Confused word: me and my
Confused word: an and and
Confused word: though and through
Confused word: there and their
There were 32 pedler confusions
The percentage of pedler confusions is 0.813421453990849%
